<img src="images/logo.png" width=180, align="center"/>

Master's degree in Intelligent Systems

Subject: 11754 - Deep Learning

Year: 2025-2026

Professor: Miguel Ángel Calafat Torrens

# LAB 4 — Diffusion Models

**In this lab you have to deliver only this file, perfectly completed, executed and showing the outputs of the cells.**

This lab builds on the concepts from LESSON 4A (theory, forward process, UNet architecture) and LESSON 4B (training, noise schedules, inference). You will complete 5 exercises that require both implementation and written analysis.

## Exercises Overview
- **Exercise A** — Architecture Analysis (no GPU, written)
- **Exercise B** — Training Analysis Experiment (moderate GPU)
- **Exercise C** — Noise Schedule Analysis (low GPU)
- **Exercise D** — Reverse Process & Partial Denoising (low GPU)
- **Exercise E** — Critical Thinking (no GPU, written)

**You can modify, add or remove any cell that you want to fulfill the requirements.**

**Written analysis guidelines:** All exercises with written components should demonstrate understanding of the underlying concepts. Minimum word counts are indicated per exercise.

In [ ]:
# Connect to your drive (Colab only)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive')
    %cd '/content/gdrive/MyDrive/LABS2026/LAB04'
    %ls -l

In [ ]:
import logging
import os
import pathlib
import sys

import torch
from torch import nn, optim
from matplotlib import pyplot as plt
import numpy as np
from tqdm import tqdm

PROJECT_DIR = str(pathlib.Path().resolve())
sys.path.append(PROJECT_DIR)

import helper_L4 as hp

logging.basicConfig(format="%(asctime)s - %(levelname)s: %(message)s",
                    level=logging.INFO, datefmt="%I:%M:%S")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# Dataset setup: extract from zip if needed
if IN_COLAB:
    dataset_zip = '/content/gdrive/MyDrive/datasets/landscape_pictures_1000.zip'
else:
    dataset_zip = os.path.join(PROJECT_DIR, '..', 'datasets', 'landscape_pictures_1000.zip')

DATASET_PATH = hp.extract_dataset(dataset_zip, remove_zip=IN_COLAB)
dataloader = hp.get_data_flat(DATASET_PATH, image_size=64, batch_size=4)

---
# Exercise A — Architecture Analysis (no GPU, written)

**Requirements:**

1. Read the `UNet` class in `helper_L4.py`. For an input of shape `(1, 3, 64, 64)` and timestep `t=500`, manually trace the tensor shape at each stage of the forward pass. Create a table: layer → input shape → output shape for: `inc`, `down1`, `sa1`, `down2`, `sa2`, `down3`, `sa3`, `bot1-3`, `up1`, `sa4`, `up2`, `sa5`, `up3`, `sa6`, `outc`.

2. Calculate total trainable parameters and break down by encoder, bottleneck, decoder.

3. Explain why **skip connections** are critical for denoising — what information flows through them that the bottleneck alone cannot preserve?

4. The SelfAttention blocks are placed at specific resolutions: in the encoder at 32×32, 16×16, 8×8; in the decoder at 16×16, 32×32, 64×64. Explain why self-attention is **NOT** placed at the 64×64 input resolution in the encoder path. What would be the computational cost?

In [ ]:
# Exercise A — Shape tracing and parameter counting
model = hp.UNet(device='cpu')

# Your code here: trace shapes and count parameters

**Exercise A — Written Analysis (minimum 200 words):**

*Write your shape table and analysis here.*

---
# Exercise B — Training Analysis Experiment (moderate GPU)

**Requirements:**

Use the `train()` function demonstrated in LESSON 4B (or your own adaptation). Run **3 training experiments**, each for **30 epochs**:

1. **Baseline**: no learning rate scheduler.
2. **CosineAnnealingLR**: use `torch.optim.lr_scheduler.CosineAnnealingLR`. Choose appropriate `T_max` and `eta_min` values.
3. **A scheduler of YOUR choice** (e.g., StepLR, ExponentialLR, OneCycleLR, ReduceLROnPlateau — you choose and justify).

For each run, **re-initialize the model from scratch**.

After all 3 runs:
- (a) Plot all 3 loss curves on the **same figure** with a legend.
- (b) Generate 4 sample images from each of the 3 trained models and display them side by side (3 columns × 4 rows).
- (c) **Written analysis (min 200 words):** compare convergence speed, final loss, and visual quality. Explain why you chose your third scheduler and how it performed.

In [ ]:
# Exercise B — Run 1: Baseline (30 epochs, no scheduler)

In [ ]:
# Exercise B — Run 2: CosineAnnealingLR (30 epochs)

In [ ]:
# Exercise B — Run 3: Your chosen scheduler (30 epochs)

In [ ]:
# Exercise B — Plot all 3 loss curves overlaid

In [ ]:
# Exercise B — Generate and compare samples from all 3 models

**Exercise B — Written Analysis (minimum 200 words):**

*Write your analysis here comparing the 3 training experiments.*

---
# Exercise C — Noise Schedule Analysis (low GPU)

**Requirements:**

1. Implement a `SigmoidDiffusion` class that subclasses `hp.Diffusion`. Override `prepare_noise_schedule` with a sigmoid function: $\beta(t) = \beta_{\text{start}} + (\beta_{\text{end}} - \beta_{\text{start}}) \cdot \sigma(s \cdot (t - 0.5))$, normalized to ensure $\beta$ spans exactly $[\beta_{\text{start}}, \beta_{\text{end}}]$. Use `sigmoid_scale=8.0`.

2. Test your implementation with `hp.test_sigmoid_diffusion()`.

3. Create a figure with **3 subplots** showing $\beta_t$, $\bar{\alpha}_t$, and $\log(\text{SNR})$ for all 3 schedules (linear via `hp.Diffusion`, cosine via `hp.CosineDiffusion`, sigmoid via your class) overlaid.

4. Using a sample image from the dataset, visualize forward diffusion at $t = [0, 200, 400, 600, 800, 999]$ for ALL 3 schedules: 3 rows × 6 columns.

5. **Written analysis (min 200 words):** which schedule preserves more information at intermediate steps? Why? What does the log(SNR) plot reveal about information decay rates?

In [ ]:
# Exercise C — SigmoidDiffusion
class SigmoidDiffusion(hp.Diffusion):
    def __init__(self, noise_steps=1000, beta_start=1e-4, beta_end=0.02,
                 sigmoid_scale=8.0, img_size=256, device="cuda"):
        pass

    def prepare_noise_schedule(self):
        pass

In [ ]:
# Exercise C — Test implementation
hp.test_sigmoid_diffusion(SigmoidDiffusion, plot=False)

In [ ]:
# Exercise C — Schedule comparison plots (3 subplots: beta_t, alpha_bar_t, log(SNR))

In [ ]:
# Exercise C — Forward diffusion visualization (3 rows × 6 cols)

**Exercise C — Written Analysis (minimum 200 words):**

*Write your analysis here about noise schedules.*

---
# Exercise D — Partial Denoising & Image Generation (low GPU)

**Requirements:**

1. Load the pretrained model (`models/unconditional_ckpt.pt`).

2. Generate 16 images and display in a 4×4 grid.

3. **Partial denoising experiment:** Take a real image from the dataset. Add noise to $t=300$, $t=500$, and $t=700$ using `diffusion.noise_images()`. Then denoise each back to $t=0$ using `diffusion.denoise_step()` in a loop. Display in a single figure: original, partially denoised from $t=300$, from $t=500$, from $t=700$, and a fully generated image (from $t = \text{noise\_steps} - 1$).

4. **Written analysis (min 200 words):** why does more noise → more "creative" output? Relate to $\bar{\alpha}_t$ values. What is the practical application of partial denoising (img2img)?

In [ ]:
# Exercise D — Load pretrained model

In [ ]:
# Exercise D — Generate 16 images (4×4 grid)

In [ ]:
# Exercise D — Partial denoising experiment

**Exercise D — Written Analysis (minimum 200 words):**

*Write your analysis here about partial denoising and its relationship to $\bar{\alpha}_t$.*

---
# Exercise E — Critical Thinking (no GPU, written)

Answer the following questions. Each answer: 100–200 words.

1. **Sampling speed**: DDPM needs $T=1000$ forward passes for ONE image. A GAN generates in a single pass. Calculate the time to generate 16 images assuming 50ms per UNet pass and 20ms per GAN pass. Research DDIM (Song et al., 2020) and explain how it reduces sampling steps. What is the key mathematical insight?

2. **ε-prediction vs x₀-prediction**: Using $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1-\bar{\alpha}_t}\, \epsilon$, show algebraically that knowing $\epsilon$ is equivalent to knowing $x_0$ (given $x_t$ and $t$). Then explain intuitively why predicting $\epsilon$ leads to more stable training.

**Exercise E — Answers:**

**1.** *Your answer here.*

**2.** *Your answer here.*